# RQ1 — Reentrant clustering

How do Watson–Crick attraction and chain stiffness affect RNA clustering?

This notebook uses **wide, condition-aggregated** CSVs. Each condition mean is
computed from independent trajectories. Final-frame XY density projections are
available as an **optional GSD visualization**, not a framewise-CSV dependency.

In [ ]:
from pathlib import Path
from functools import partial
from dataclasses import replace
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Works when Jupyter starts from the repository or analysis/notebooks/.
here = Path.cwd().resolve()
ROOT = next(
    (p for p in (here, *here.parents)
     if (p / "analysis" / "src" / "plots" / "style.py").is_file()),
    None
)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from inside the repository checkout.")
sys.path.insert(0, str(ROOT / "analysis"))

from src.plots.style import PlotStyle
from src.plots.data import load_sweep, filter_rows
from src.plots.panels import panels_by

STYLE = PlotStyle()
OUTPUT_ROOT = ROOT / "analysis_results"


In [ ]:
from src.plots.figures import (
    plot_cluster_bins, plot_metric_series, plot_metric_points
)
SWEEP = "50b_reentrant_full"
summary = load_sweep(OUTPUT_ROOT, SWEEP, "summary")
replicates = load_sweep(OUTPUT_ROOT, SWEEP, "replicates")
DENSITY = 0.01
summary = filter_rows(summary, density=DENSITY)
replicates = filter_rows(replicates, density=DENSITY)
print(f"{len(replicates)} trajectories, {len(summary)} conditions")

## Cluster-size composition

For RQ1, cluster-size classes are defined by **number of molecules**;
bar heights are fractions of the system's particles (equivalent to molecule
fractions for equal-length RNAs).

In [ ]:
plot_fn = partial(plot_cluster_bins, scheme="molecule_count", style=STYLE)
fig, axes = panels_by(
    summary, by="angle_k_ppp", plot_fn=plot_fn, style=STYLE,
    ncols=3, title="Reentrant cluster-size redistribution",
)
plt.show()

## Intra- versus intermolecular contacts

Mean base-pair counts show where accepted interactions are allocated.

In [ ]:
fig, axes = panels_by(
    summary, by="angle_k_ppp",
    plot_fn=partial(plot_metric_series, metric="n_inter_contacts",
                    style=STYLE, color="#286D9E", label="Intermolecular"),
    style=STYLE, title="Intermolecular contacts across attraction",
)
plt.show()
fig, axes = panels_by(
    summary, by="angle_k_ppp",
    plot_fn=partial(plot_metric_series, metric="n_intra_contacts",
                    style=STYLE, color="#C36C43", label="Intramolecular"),
    style=STYLE, title="Intramolecular contacts across attraction",
)
plt.show()

## Molecular graph connectivity

Qualifying graph edges require at least two particle contacts per molecular
pair. Mean edge weight and edge count are distinct observables.

In [ ]:
for metric in ("n_edges", "mean_edge_weight", "molecule_recurrence"):
    fig, axes = panels_by(
        summary, by="angle_k_ppp",
        plot_fn=partial(plot_metric_series, metric=metric, style=STYLE,
                        band="std"),
        style=STYLE, title=STYLE.label(metric),
    )
    plt.show()

## Representative final-frame XY projections

Optional: these read one **final GSD frame per parameter run**, unlike the
CSV-only plots above. They follow the reentrance grid: stiffness by attraction.
No all-plane profiles or extra length-specific diagnostics.

In [ ]:
SHOW_PROJECTIONS = False  # Set True only when original GSDs are available.
if SHOW_PROJECTIONS:
    from src.plots.projections import plot_projection_sweep
    GSD_ROOT = ROOT.parent / "validation" / "bulk" / "output_dir"
    fig, axes = plot_projection_sweep(
        GSD_ROOT / SWEEP, row_by="angle_k_ppp",
        density=DENSITY, style=STYLE, attractions=(2, 4, 8),
        title="RQ1 · reentrant RNA clustering",
    )
    plt.show()

## Optional interactive metric explorer

Choose a metric, density, and stiffness interactively. Requires `ipywidgets`;
the normal plots above do not depend on it.

In [ ]:
SHOW_EXPLORER = False
if SHOW_EXPLORER:
    from src.plots.explorer import summary_explorer
    summary_explorer(load_sweep(OUTPUT_ROOT, SWEEP, "summary"), style=STYLE)


## Interpretation

Compare low, intermediate, and high attraction: formation of large assemblies,
redistribution toward smaller clusters, and the distinction between numbers
of graph edges and the weight of each molecular connection.